# 06 — Recommendation System

## Purpose

This notebook builds and evaluates the recommendation layer for the Olist e-commerce intelligence platform.

The recommendation strategy is intentionally incremental:

```text
Popularity Baseline
        ↓
Item-Item Collaborative Filtering
        ↓
Customer Candidate Generation
        ↓
Precision@K / Recall@K / Hit Rate@K
        ↓
Reusable Recommendation Artifact
```

The evaluation is **time-aware**: historical purchases are used to build recommendations and future purchases are used to evaluate them.

## Business questions

- What products are generally popular?
- Which products are commonly purchased together?
- Given a customer's history, which unseen products are good candidates?
- Does personalized recommendation outperform a simple popularity baseline?
- Can recommendation candidates be generated efficiently for the API?

### Important

This notebook uses purchase interactions as implicit feedback.

A purchase is treated as a positive interaction. There is no explicit negative-feedback label in the Olist data.

In [1]:
from pathlib import Path
import sys
import warnings
from collections import defaultdict

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sqlalchemy import text

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: d:\ecommerce-intelligence


## 1. Load purchase interactions from Supabase

In [2]:
from src.database import get_engine

engine = get_engine()

interaction_query = text("""
SELECT
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    oi.product_id,
    oi.price,
    oi.freight_value
FROM olist_customers c
JOIN olist_orders o
    ON c.customer_id = o.customer_id
JOIN olist_order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
ORDER BY o.order_purchase_timestamp
""")

with engine.connect() as connection:
    interactions = pd.read_sql(
        interaction_query,
        connection,
    )

interactions["order_purchase_timestamp"] = pd.to_datetime(
    interactions["order_purchase_timestamp"],
    errors="coerce",
)

interactions["price"] = pd.to_numeric(
    interactions["price"],
    errors="coerce",
).fillna(0)

interactions["freight_value"] = pd.to_numeric(
    interactions["freight_value"],
    errors="coerce",
).fillna(0)

interactions = interactions.dropna(
    subset=[
        "customer_unique_id",
        "product_id",
        "order_purchase_timestamp",
    ]
)

print("Interaction rows:", f"{len(interactions):,}")
print("Customers:", f"{interactions['customer_unique_id'].nunique():,}")
print("Products:", f"{interactions['product_id'].nunique():,}")
print("Orders:", f"{interactions['order_id'].nunique():,}")

display(interactions.head())

Interaction rows: 112,101
Customers: 94,983
Products: 32,729
Orders: 98,199


,customer_unique_id,order_id,order_purchase_timestamp,product_id,price,freight_value
0,b7d76e111c89f7ebf14761390f0f7d17,2e7a8482f6fb09756ca50c10d7bfc047,2016-09-04 21:15:19,c1488892604e4ba5cff5b4eb4d595400,39.99,31.67
1,b7d76e111c89f7ebf14761390f0f7d17,2e7a8482f6fb09756ca50c10d7bfc047,2016-09-04 21:15:19,f293394c72c9b5fafd7023301fc21fc2,32.90,31.67
2,830d5b7aaa3b6f1e9ad63703bec97d23,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83
3,830d5b7aaa3b6f1e9ad63703bec97d23,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83
4,830d5b7aaa3b6f1e9ad63703bec97d23,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83


## 2. Create a temporal train/test split

We use the 80th percentile of purchase time as the cutoff.

```text
Historical 80%
    ↓
Recommendation training

Future 20%
    ↓
Recommendation evaluation
```

This avoids evaluating a recommender using future interactions that were already visible during training.

In [3]:
cutoff = interactions["order_purchase_timestamp"].quantile(0.80)

train_interactions = interactions[
    interactions["order_purchase_timestamp"] <= cutoff
].copy()

test_interactions = interactions[
    interactions["order_purchase_timestamp"] > cutoff
].copy()

print("Cutoff:", cutoff)
print("Train interactions:", f"{len(train_interactions):,}")
print("Test interactions:", f"{len(test_interactions):,}")
print(
    "Train customers:",
    f"{train_interactions['customer_unique_id'].nunique():,}",
)
print(
    "Test customers:",
    f"{test_interactions['customer_unique_id'].nunique():,}",
)

Cutoff: 2018-05-24 10:43:37
Train interactions: 89,681
Test interactions: 22,420
Train customers: 75,922
Test customers: 19,527


## 3. Build the popularity baseline

In [4]:
popularity = (
    train_interactions
    .groupby("product_id")
    .agg(
        purchase_count=("order_id", "nunique"),
        customer_count=("customer_unique_id", "nunique"),
        revenue=("price", "sum"),
    )
    .sort_values(
        ["purchase_count", "customer_count"],
        ascending=False,
    )
    .reset_index()
)

display(popularity.head(20))

fig = px.bar(
    popularity.head(20).sort_values("purchase_count"),
    x="purchase_count",
    y="product_id",
    orientation="h",
    title="Top Products — Popularity Baseline",
    labels={
        "purchase_count": "Purchases",
        "product_id": "Product",
    },
)
fig.show()

,product_id,purchase_count,customer_count,revenue
0,99a4788cb24856965c36a24e339b6058,423,422,39150.16
1,aca2eb7d00ea1a7b8ebd4e68314663af,417,416,36240.60
2,422879e10f46682990de24d770e7f83d,333,330,24980.30
3,53b36df67ebb7c41585e8d54d6772e08,285,282,35198.42
4,389d119b48cf3043d311335e499d9c6b,284,283,19843.79
5,d1c427060a0f73f6b889a5c7c61f2ac4,276,276,41430.94
6,368c6c730842d78016ad823897a372db,267,265,19509.90
7,53759a2ecddad2bb87a079a1f1519f73,265,261,18840.30
8,154e7e31ebfa092203795c972e5804a6,261,260,6133.27
9,2b4609f8948be18874494203496bc318,227,225,20127.72


## 4. Build customer purchase histories

For each customer we create the set of products purchased during training.

These histories are the basis for personalized recommendations.

In [5]:
customer_history = (
    train_interactions
    .groupby("customer_unique_id")["product_id"]
    .apply(set)
    .to_dict()
)

print(
    "Customers with training history:",
    f"{len(customer_history):,}",
)

sample_customer = next(iter(customer_history))

print("Example customer:", sample_customer)
print(
    "Products purchased:",
    len(customer_history[sample_customer]),
)

Customers with training history: 75,922
Example customer: 0000366f3b9a7992bf8c76cfdf3221e2
Products purchased: 1


## 5. Build item-item co-purchase counts

Two products are considered related when they appear in the same order.

For example:

```text
Order A
 ├── Product 1
 ├── Product 2
 └── Product 3

Creates:
Product 1 ↔ Product 2
Product 1 ↔ Product 3
Product 2 ↔ Product 3
```

We use these relationships to recommend products similar to a customer's previous purchases.

In [6]:
pair_counts = defaultdict(int)
product_order_counts = defaultdict(int)

for order_id, group in train_interactions.groupby("order_id"):
    products = sorted(group["product_id"].unique())

    for product in products:
        product_order_counts[product] += 1

    for i in range(len(products)):
        for j in range(i + 1, len(products)):
            a = products[i]
            b = products[j]
            pair_counts[(a, b)] += 1

print("Unique product pairs:", f"{len(pair_counts):,}")

Unique product pairs: 3,154


## 6. Convert co-purchases into normalized similarity

Raw co-purchase count favors products that are already extremely popular.

We therefore calculate a Jaccard-style similarity:

```text
co_purchase(A,B)
------------------------------
orders(A) + orders(B) - co_purchase(A,B)
```

The resulting score is easier to compare across product pairs.

In [7]:
item_similarity = defaultdict(dict)

for (a, b), co_count in pair_counts.items():
    denominator = (
        product_order_counts[a]
        + product_order_counts[b]
        - co_count
    )

    if denominator <= 0:
        continue

    similarity = co_count / denominator

    item_similarity[a][b] = similarity
    item_similarity[b][a] = similarity

print(
    "Products with similarity relationships:",
    f"{len(item_similarity):,}",
)

Products with similarity relationships: 3,926


## 7. Recommendation function

The personalized recommender:

1. takes the customer's historical products,
2. finds related products,
3. scores unseen candidates,
4. combines similarity and popularity,
5. returns the top K products.

Popularity is used as a tie-breaker so recommendations remain stable when similarity evidence is weak.

In [8]:
popularity_rank = {
    row.product_id: int(row.Index)
    for row in popularity.reset_index().itertuples()
}

popularity_score = {}

max_count = (
    popularity["purchase_count"].max()
    if not popularity.empty
    else 1
)

for row in popularity.itertuples():
    popularity_score[row.product_id] = (
        row.purchase_count / max_count
    )


def recommend_for_customer(
    customer_id: str,
    k: int = 10,
) -> list[str]:
    """Recommend unseen products using item-item similarity."""

    history = customer_history.get(customer_id, set())

    if not history:
        return popularity.head(k)["product_id"].tolist()

    candidate_scores = defaultdict(float)

    for purchased_product in history:
        related = item_similarity.get(
            purchased_product,
            {},
        )

        for candidate, similarity in related.items():
            if candidate in history:
                continue

            candidate_scores[candidate] += similarity

    if not candidate_scores:
        return popularity.head(k)["product_id"].tolist()

    ranked_candidates = sorted(
        candidate_scores.items(),
        key=lambda item: (
            item[1],
            popularity_score.get(item[0], 0),
        ),
        reverse=True,
    )

    return [
        product_id
        for product_id, _ in ranked_candidates[:k]
    ]


print(
    "Example recommendations:",
    recommend_for_customer(sample_customer, k=10),
)

Example recommendations: ['2b10e945dae5434075c8bb2be0d17325', '3ea30fb7b4c6d17f44c1594a713c224c', 'eb53f94fdc60278efcef123bb275658a', '42155695adbe665066ad812855fe523a', '525947dbe3304ac32bf51602f9557c12']


## 8. Evaluate the popularity baseline

For each test customer:

- recommendations are generated from training-period popularity,
- future test purchases are the ground truth,
- recommendations are evaluated using Hit Rate and Recall.

Only customers who appear in both train and test are evaluated.

In [9]:
test_truth = (
    test_interactions
    .groupby("customer_unique_id")["product_id"]
    .apply(set)
    .to_dict()
)

eligible_customers = [
    customer_id
    for customer_id in test_truth
    if customer_id in customer_history
]

print(
    "Eligible evaluation customers:",
    f"{len(eligible_customers):,}",
)

Eligible evaluation customers: 466


## 9. Evaluation metrics

In [10]:
def evaluate_recommendations(
    recommendation_function,
    customers,
    truth,
    k: int = 10,
):
    hits = 0
    total_relevant = 0
    total_recommended_relevant = 0

    for customer_id in customers:
        actual = truth.get(customer_id, set())

        if not actual:
            continue

        recommended = set(
            recommendation_function(
                customer_id,
                k=k,
            )
        )

        overlap = recommended.intersection(actual)

        if overlap:
            hits += 1

        total_relevant += len(actual)
        total_recommended_relevant += len(overlap)

    customer_count = len(customers)

    hit_rate = (
        hits / customer_count
        if customer_count
        else 0
    )

    recall = (
        total_recommended_relevant / total_relevant
        if total_relevant
        else 0
    )

    precision = (
        total_recommended_relevant
        / (customer_count * k)
        if customer_count
        else 0
    )

    return {
        "precision_at_k": precision,
        "recall_at_k": recall,
        "hit_rate_at_k": hit_rate,
        "customers_evaluated": customer_count,
    }


def popularity_recommender(
    customer_id: str,
    k: int = 10,
) -> list[str]:
    history = customer_history.get(
        customer_id,
        set(),
    )

    return [
        product_id
        for product_id in popularity["product_id"]
        if product_id not in history
    ][:k]


baseline_metrics = evaluate_recommendations(
    popularity_recommender,
    eligible_customers,
    test_truth,
    k=10,
)

display(
    pd.DataFrame([baseline_metrics])
)

,precision_at_k,recall_at_k,hit_rate_at_k,customers_evaluated
0,0.001502,0.013487,0.015021,466


## 10. Evaluate personalized item-item recommendations

In [11]:
personalized_metrics = evaluate_recommendations(
    recommend_for_customer,
    eligible_customers,
    test_truth,
    k=10,
)

display(
    pd.DataFrame([personalized_metrics])
)

,precision_at_k,recall_at_k,hit_rate_at_k,customers_evaluated
0,0.001073,0.009634,0.01073,466


## 11. Compare recommendation approaches

In [12]:
comparison = pd.DataFrame([
    {
        "method": "Popularity Baseline",
        **baseline_metrics,
    },
    {
        "method": "Item-Item Collaborative Filtering",
        **personalized_metrics,
    },
])

display(
    comparison.round(4)
)

metric_plot = comparison.melt(
    id_vars="method",
    value_vars=[
        "precision_at_k",
        "recall_at_k",
        "hit_rate_at_k",
    ],
    var_name="metric",
    value_name="score",
)

fig = px.bar(
    metric_plot,
    x="metric",
    y="score",
    color="method",
    barmode="group",
    title="Recommendation Model Comparison @ K=10",
)
fig.show()

,method,precision_at_k,recall_at_k,hit_rate_at_k,customers_evaluated
0,Popularity Baseline,0.0015,0.0135,0.0150,466
1,Item-Item Collaborative Filtering,0.0011,0.0096,0.0107,466


## 12. Generate recommendation candidates

Create a reusable recommendation table for customers in the evaluation population.

This table can later be joined with product metadata and exposed through FastAPI.

In [13]:
recommendation_rows = []

for customer_id in eligible_customers:
    recommendations = recommend_for_customer(
        customer_id,
        k=10,
    )

    for rank, product_id in enumerate(
        recommendations,
        start=1,
    ):
        recommendation_rows.append({
            "customer_unique_id": customer_id,
            "product_id": product_id,
            "rank": rank,
        })

recommendation_df = pd.DataFrame(
    recommendation_rows
)

print(
    "Recommendation rows:",
    f"{len(recommendation_df):,}",
)

display(recommendation_df.head(20))

Recommendation rows: 3,640


,customer_unique_id,product_id,rank
0,004b45ec5c64187465168251cd1c9c2f,0bcc3eeca39e1064258aa1e932269894,1
1,004b45ec5c64187465168251cd1c9c2f,422879e10f46682990de24d770e7f83d,2
2,004b45ec5c64187465168251cd1c9c2f,53759a2ecddad2bb87a079a1f1519f73,3
3,004b45ec5c64187465168251cd1c9c2f,368c6c730842d78016ad823897a372db,4
4,004b45ec5c64187465168251cd1c9c2f,389d119b48cf3043d311335e499d9c6b,5
5,00a39521eb40f7012db50455bf083460,4d69476a7bd01b8eec1c87e60778364c,1
6,00a39521eb40f7012db50455bf083460,cec09725da5ed01471d9a505e7389d37,2
7,012a218df8995d3ec3bb221828360c86,99a4788cb24856965c36a24e339b6058,1
8,012a218df8995d3ec3bb221828360c86,aca2eb7d00ea1a7b8ebd4e68314663af,2
9,012a218df8995d3ec3bb221828360c86,422879e10f46682990de24d770e7f83d,3


## 13. Attach product category information

Adding category information makes recommendations easier to interpret in the dashboard and AI insight layer.

In [ ]:
category_query = text("""
SELECT
    p.product_id,
    COALESCE(
        t.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS category
FROM olist_products p
LEFT JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
""")

try:
    with engine.connect() as connection:
        product_categories = pd.read_sql(
            category_query,
            connection,
        )
except Exception as e:
    print(f"Database connection error: {e}")
    print("Creating empty product_categories dataframe as fallback...")
    product_categories = pd.DataFrame({
        'product_id': recommendation_df['product_id'].unique(),
        'category': 'Unknown'
    })

recommendation_df = recommendation_df.merge(
    product_categories,
    on="product_id",
    how="left",
)

display(recommendation_df.head(20))

OperationalError: (psycopg2.OperationalError) could not translate host name "db.njljkgmxjhvbnevbegoc.supabase.co" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 14. Save recommendation artifacts

The serialized artifact contains:

- customer histories,
- item-item similarity,
- popularity ranking,
- popularity scores.

This lets the application generate recommendations without rebuilding the full interaction matrix on every API request.

In [ ]:
ARTIFACT_PATH = (
    MODEL_DIR
    / "recommendation_artifact.joblib"
)

RECOMMENDATION_PATH = (
    OUTPUT_DIR
    / "recommendations.csv"
)

artifact = {
    "customer_history": customer_history,
    "item_similarity": dict(item_similarity),
    "popularity": popularity,
    "popularity_score": popularity_score,
    "top_k_default": 10,
}

joblib.dump(
    artifact,
    ARTIFACT_PATH,
)

recommendation_df.to_csv(
    RECOMMENDATION_PATH,
    index=False,
)

comparison.to_csv(
    OUTPUT_DIR / "recommendation_model_comparison.csv",
    index=False,
)

print(f"Saved recommendation artifact: {ARTIFACT_PATH}")
print(f"Saved recommendations: {RECOMMENDATION_PATH}")

# Final validation

Expected artifacts:

```text
models/
└── recommendation_artifact.joblib

data/processed/
├── recommendations.csv
└── recommendation_model_comparison.csv
```

### Evaluation principle

The important portfolio result is not simply:

> "We built a recommender."

It is:

> **"We compared a personalized recommendation method against a popularity baseline using future customer interactions as the evaluation target."**

If personalized performance does not beat the baseline, keep that result honest and investigate why rather than artificially tuning the evaluation.